In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
import cv2
from qrdet import QRDetector
from tqdm import tqdm

# =========================
# PATH
# =========================

BASE = '/content/drive/MyDrive/AI_BaiTapLon/cccd_extraction_src'

INPUT_DIR = os.path.join(BASE, 'data/raw/front_315')
OUTPUT_DIR = os.path.join(BASE, 'data/raw/front_315_rotated')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# QR DETECTOR
# =========================

detector = QRDetector()

# =========================
# GET IMAGE LIST
# =========================

image_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print(f'Tìm thấy {len(image_files)} ảnh')

# =========================
# PROCESS
# =========================

success = 0
failed = 0

for file_name in tqdm(image_files):

    try:

        input_path = os.path.join(INPUT_DIR, file_name)
        output_path = os.path.join(OUTPUT_DIR, file_name)

        image = cv2.imread(input_path)

        if image is None:
            failed += 1
            continue

        h, w = image.shape[:2]

        # =========================
        # DETECT QR
        # =========================

        detections = detector.detect(image=image)

        # Không detect được QR
        if len(detections) == 0:

            # lưu nguyên ảnh
            cv2.imwrite(output_path, image)

            failed += 1
            continue

        detection = detections[0]

        x1, y1, x2, y2 = detection["bbox_xyxy"]
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        qr_center_x = (x1 + x2) / 2
        qr_center_y = (y1 + y2) / 2

        # =========================
        # XOAY ẢNH
        # QR chuẩn:
        # nằm góc trên bên phải
        # =========================

        # Đúng chiều
        if qr_center_x > w/2 and qr_center_y < h/2:

            image_rotated = image.copy()

        # Ngược 180
        elif qr_center_x < w/2 and qr_center_y > h/2:

            image_rotated = cv2.rotate(
                image,
                cv2.ROTATE_180
            )

        # Xoay phải
        elif qr_center_x < w/2 and qr_center_y < h/2:

            image_rotated = cv2.rotate(
                image,
                cv2.ROTATE_90_CLOCKWISE
            )

        # Xoay trái
        else:

            image_rotated = cv2.rotate(
                image,
                cv2.ROTATE_90_COUNTERCLOCKWISE
            )

        # =========================
        # SAVE
        # =========================

        cv2.imwrite(output_path, image_rotated)

        success += 1

    except Exception as e:

        print(f'Lỗi {file_name}: {e}')
        failed += 1

# =========================
# DONE
# =========================

print('\n===== DONE =====')
print(f'Success: {success}')
print(f'Failed : {failed}')
print(f'Saved to: {OUTPUT_DIR}')

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/qrdet/qrdet.py:129: UserWarning: QRDetector has been updated to use the new YoloV8 model. Use legacy=True when calling detect for backwards compatibility with 1.x versions. Or update to new output (new output is a tuple of dicts, containing several new information (1.x output is accessible through 'bbox_xyxy' and 'confidence').Forget this message if you are reading it from QReader. [This is a first download warning and will be removed at 2.1]
  warn("QRDetector has been updated to use the new YoloV8 model. Use legacy=True when calling detect "


Tìm thấy 231 ảnh


100%|██████████| 231/231 [02:58<00:00,  1.30it/s]


===== DONE =====
Success: 231
Failed : 0
Saved to: /content/drive/MyDrive/AI_BaiTapLon/cccd_extraction_src/data/raw/front_315_rotated


In [2]:
!pip install qrdet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.9 MB/s eta 0:00:00
